# 14. The daily report that is five and a half hours off

**Track:** debug-the-ai | **Difficulty:** medium | **Tags:** time zones, utc, ist, dates, daily aggregation

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("014-utc-dates-in-india-report")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "app_events": """
UEFSMRUEFeAlFZATTBXcBBUAEgAA8BIIQZwABQEAQg0IAEMNCABEDQgARQ0IAEYNCABHDQgASA0IAEkNCABKDQgASw0IAEwNCABN
DQgATg0IAE8NCABQDQgAUQ0IAFINCABTDQgAVA0IAFUNCABWDQgAVw0IAFgNCABZDQgAWg0IAFsNCABcDQgAXQ0IAF4NCABfDQgA
YA0IAGENCABiDQgAYw0IAGQNCABlDQgAZg0IAGcNCABoDQgAaQ0IAGoNCABrDQgAbA0IAG0NCABuDQgAbw0IAHANCABxDQgAcg0I
AHMNCAB0DQgAdQ0IAHYNCAB3DQgAeA0IAHkNCAB6DQgAew0IAHwNCAB9DQgAfg0IAH8NCACADQgAgQ0IAIINCACDDQgAhA0IAIUN
CACGDQgAhw0IAIgNCACJDQgAig0IAIsNCACMDQgAjQ0IAI4NCACPDQgAkA0IAJENCACSDQgAkw0IAJQNCACVDQgAlg0IAJcNCACY
DQgAmQ0IAJoNCACbDQgAnA0IAJ0NCACeDQgAnw0IAKANCAChDQgAog0IAKMNCACkDQgApQ0IAKYNCACnDQgAqA0IAKkNCACqDQgA
qw0IAKwNCACtDQgArg0IAK8NCACwDQgAsQ0IALINCACzDQgAtA0IALUNCAC2DQgAtw0IALgNCAC5DQgAug0IBLucZc8EALwNCAC9
DQgAvg0IAL8NCADADQgAwQ0IAMINCADDDQgAxA0IAMUNCADGDQgAxw0IAMgNCADJDQgAyg0IAMsNCADMDQgAzQ0IAM4NCADPDQgA
0A0IANENCADSDQgA0w0IANQNCADVDQgA1g0IANcNCADYDQgA2Q0IANoNCADbDQgA3A0IAN0NCADeDQgA3w0IAOANCADhDQgA4g0I
AOMNCADkDQgA5Q0IAOYNCADnDQgA6A0IAOkNCADqDQgA6w0IAOwNCADtDQgA7g0IAO8NCADwDQgA8Q0IAPINCADzDQgA9A0IAPUN
CAD2DQgA9w0IAPgNCAD5DQgA+g0IAPsNCAD8DQgA/Q0IAP4NCAD/DQgEAJ1JKAABDQgAAg0IAAMNCAAEDQgABQ0IAAYNCAAHDQgA
CA0IAAkNCAAKDQgACw0IAAwNCAANDQgADg0IAA8NCAAQDQgAEQ0IABINCAATDQgAFA0IABUNCAAWDQgAFw0IABgNCAAZDQgAGg0I
ABsNCAAcDQgAHQ0IAB4NCAAfDQgAIA0IACENCAAiDQgAIw0IACQNCAAlDQgAJg0IACcNCAAoDQgAKQ0IACoNCAArDQgALA0IAC0N
CAAuDQgALw0IADANCAAxDQgAMg0IADMNCAA0DQgANQ0IADYNCAA3DQgAOA0IADkNCAA6DQgAOw0IADwNCAA9DQgAPg0IAD8NCABA
DQgAQQ0IAEINCABDDQgARA0IAEUNCABGDQgARw0IAEgNCABJDQgASg0IAEsNCABMDQgATQ0IAE4NCABPDQgAUA0IAFENCABSDQgA
Uw0IAFQNCABVDQgAVg0IAFcNCABYDQgAWQ0IAFoNCABbDQgAXA0IAF0NCABeDQgAXw0IAGANCABhDQgAYg0IAGMNCABkDQgAZQ0I
AGYNCABnDQgAaA0IAGkNCABqDQgAaw0IAGwNCDxtnQAAAAAAAG6dAAAAAAAAFQAVvgUVyAUsFdwEFRAVBhUGHBgIbp0AAAAAAAAY
CEGcAAAAAAAAFgAoCG6dAAAAAAAAGAhBnAAAAAAAABERAAAA3wL0XgEDAAAA3AQBCU0AAggYQKCAgQMIEihYwKCBgwcQIkiYQKGC
hQsYMmjYwKGDhw8gQogYQaKEiRMoUqhYwaKFixcwYsiYQaOGjRs4cujYwaOHjx9AgggZQqSIkSNIkihZwqSJkydQokiZQqWKlStY
smjZwqWLly9gwogZQ6aMmTNo0qhZw6aNmzdw4siZQ6eOnTt48ujZw6ePnz+AAgkaRKiQoUOIEilaxKiRo0eQIkmaRKmSpUuYMmna
xKmTp0+gQokaRaqUqVOoUqlaxaqVq1ewYsmaRauWrVu4cunaxauXr1/AggkbRqyYsWPIkilbxqyZs2fQokmbRq2atWvYsmnbxq2b
t2/gwokbR66cuXPo0qlbx66du3fw4smbR6+evXv48unbx6+fv38AAwocSLCgwYMIEypcyLChw4cQI0qcSLGixYsYM2rcyLGjx48g
Q4ocSbKkyZMoU6pcybIlAAAVBBXgBxWQBEwVfBUAEgAA8AMEIAAJAQQdAAkBAC0JBwQALA0IACENCAAzDQgABQ0IACUNCAAZDQgA
MQ0IBDoACQEACAkHBAAODQgAIw0IADkNCAADDQgAHA0IAA0NCAASDQgAJw0IAAwNCAAoDQgANg0IABoNCAAuDQgAFg0IADgNCAAU
DQgAPA0IAB4NCAAqDQgAJg0IAA8NCAAHDQgALw0IACQNCAA7DQgAEA0IADANCAAXDQgABA0IABMNCAAGDQgACg0IABENCAArDQgA
PQ0IADQNCAApDQgANQ0IAAsNCAA3DQgACQ0IABUNCAAiDQgAAQ0IADINCAAYDQgAHw0IAAINCDwbAAAAAAAAAD4AAAAAAAAAFQAV
2gMV4gMsFdwEFRAVBhUGHBgIPgAAAAAAAAAYCAEAAAAAAAAAFgAoCD4AAAAAAAAAGAgBAAAAAAAAABERAAAA7QHw7AMAAADcBAEG
TUAgDEQxGAeSKAvTOA+kRNJEVZV1URiRaVvWYAQXJF00ed8HdqEUXmIwGiRwlOa5ANyApsJjBKHarJYlPGzrggOpdS/8xpxZTYXZ
MZocDdl8fTQSolcyMTViV4u5ZXQwdSncbQBSzYI2aJ2wMAC6mBTMGvfBBIacgEuUlBXisJ1ssRueI7qm7nfxZY/YSPyoCCSi6Vd3
QEycyRez2iQvWPc48hJ2t+goAGd/Cls5AMzYhGtFf+UhvSr+SvPVVQ3ule1wxwKTGqgnkBOdsyWGEEkYyoXEqDa9CJyeQEnrTJJ5
euoLABUEFeAlFfohTBXcBBUAEgAA8BLQgGtsvosvBgCAvTECTy8GAAD5euyCLwYAQC+TLp4vBgCAd3t3hS8GAAAfW6GVLwYAgHbE
yH0BCCzsQTClLwYAwOOZKFoBCEi3gFnDLwYAABnTfbAvBgBAvLCDAUAQwNuOcXkBECxHutRULwYAAFV5C2QBCIhfJml3LwYAQMPU
lU0vBgCA9GV5qC8GAAB5A2e5LwYAwN2zTwE4DIAbIDEFCAiCvksBqAzA1QOSAShMgGZmcrovBgAAza1GbS8GAMDWWcYBIAxAMcYz
AZgMgFQR9AUQDJJRO5gBIHCo87+ALwYAAI8wO5MvBgBARjMltC8GAMDbSUVjLwGwSLVkaC8GAAA1KJJYLwYAQMxaHpoBCPBSac4j
ay8GAIA6o5d1LwYAQDPd4WAvBgAAgGNfQS8GAIDV1hCDLwYAABt5ZWIvBgAA4pQ7hi8GAMDqdNawLwYAAMrwzJwvBgAAvwjypi8G
AABaU70BOAzAKEl/IQgMgHRsOAF4LMBOb+5dLwYAwC6RcgWYTBTrcEQvBgAArW5Qny8GAMCZMMq2BQgkXfu3LwYAgH7xRQEgDIAI
TtcFEAiiZdQFKBDOhH2sLyFYKMZuRi8GAMCd4g/AAQgo0BddjS8GAAAn+E0B4AwAA5DTATAMANs2VgW4LHvMq14vBgCAib8SwQEI
aN9gCGYvBgBAeH4aoS8GAIA0ov5WLwYAQK7OqAGgLABUT3qxLwYAgBnF5CHIDMAcZmABQEwAe+HhSi8GAAAPsyt4LwYAQE4h9wUI
CMvs5QXwLPeRi2cvBgAAMhPkoAEIbFLQOa8vBgBAp5dzfS8GAMBX+ld0LwYAQM2TbWEBCAhL8PEheCzAFW92cy8GAIBkD3ABEAyA
1zZHIeAMwNwERwFYDEDhTzEFQCwj4JOQLwYAQKEIQbMBCEiEddiFLwYAwPF556IvBgBA6zzRAfgswE36pkMvBgDAaox4QQAwANCz
SIovBgAAf9fFmQEICCoRWCFAUIAZOvU9LwYAwHjjbFAvBgAAQBTJhAEICENsfgGoDAB8HyQlGChFyUi1LwYAgM6dtkWQCF0dtwH4
DAB2tgpFQAjuH7QBwCxAzHOwvC8GAEB04RMhaAwArkFNCTgIE2d+ARgIGUc+AWgMQAKMogHADMD/PU8hKDTA6WwCpy8GAMCva8WX
L2GYiJC2ri8GAEDaKfGyLwYAAJUTT0svBgDAy8IHWy8GAICN7hfAAQgoacSovi8GAMCn/W4luAhDuVAhKAwAFIU3BfgIvHGRBYBI
WeJxpi8GAEDtZGJOLwYAAFRXFyGoDID+2jFBIAxAJEcUAVgMwCxJLiUACK5ZNgGoMICFLW6/LwYAQIXDZnYBCAjwQ6tB4AwAYJxH
JWgsEjxDVS8GAAAALCXEAQgIwHprAUgMAGYmkAVYCIs1vSHoDIAXrZ8F0AgKSoch2AyAIYANIdgMABc1KkFgDACK8+kB6CyAoeSK
XC8GAMBPqnQByAzAxkerhXgI+zHJQagMALGIHwEoDACtfqUhYAzA1jtKAWgMAAj3kEH4DMBac05hSAwAzVagIbgMwP+8PEHIDABr
fRNhsBDAo3nofAFgCHTMYmEADMBdIzhhOLCAnqKroy8GAIB2uM+RLwYAAEJy0XEvBgDAxoqggS8GAEA3WlB2LwYAALntOJUBCAhd
ZCFBOBBAfAS0PAEQCFqsmgFgEAB+i1yHARAoauESbC8GAEDo0QpBmCzAs7LFqy8GAIDR9NSliAg1aNQlgAiMklAhkAyAdY41ASgM
gHxlfkHQDIAOO78lACjrJQSkLwYAwP94p6EYEEAwVTZHARAIIUjBQXgMAMKHTkHIDMB2o9cBWAxA5/DvIRgsgIGF6kkvBgBAqJV8
IRAQQO7MGW8BEAhoUKgB8AwAV8WZIeAsQPOdjkwvBgDADHv+JegMZYrfQgEQCDNb2IVoCFhqHiEwDIDyZrQB2AyAbZcdQbAMQPLK
owHwLMBtwEKPLwYAQFIIHUXQCLBpvCF4DACWbw2hkAxAljp9ZRgIT1TXobgMwJRU4QEYDAD7HIphQAxARO88oegMQDGrfAHoDADi
V75BWHCA4kzhUi8GAMAihgN3LwYAQJLT5JQvBgCA7jYhuAEICMD0DkFQDAAhF1ZBkEyAfwoPuy8GAAD8AyZtLwYAQLUvSkGQDICo
PYbBYAwAmglYYdAMQFBuJQG4DIBpVpVhICyA78yXqi8GAADajc5FqChTOzJNLwYAgBx3BWGQLEDpBgdwLwYAQOXXCgFwDICtiP7F
IAiCEHEF+Ij5cxtILwYAgMGPZmovBgDAousOQC8GAIAIbmZFLwYAAB5wqQEYLIDfld9ZLwYAwJ8Y0UHACMAxq0mQCE1mlaXACKFM
WwF4DADYuR5B6AyAf1fYwWgMgAP5gUEQDEBj6wZhgAyAONgbJTgIC8o5IegMAJlb5kFoDABGbY0BQDBAR1jMXy8GAEBDI0+JAQgM
K5WtmQEICL1l+wEgDABOmYUlgAgQ6l7hwAxAZ+BBQcgMgH9MdkFgDADI7B2lgAjfXLUBWAyAx358gXgMAA0PYEGwDMDEr9gFOAjS
x4qB0AwAPcgVgYAMgKbS5IEIDIBRaj4BoAzAH6sNQcAQALprtHIBgAgcvgYlyAiThkBlaAgclNghOCxAiR1vWy8GAMCBUygBGDDA
xo+KiC8GAEDvqFzCAQgIJIupIagMwLpRcgFIDMBQA24B2AwAzkk+IdAMgLOkTgXQCO3MpCGoDMCTnTIl8AilokQhWAwAI8roAVgs
QMlCmT8vBgCA226AIQgMgNh8LwFIDMB3zhHFGAiLYz4BEAyA5nVlJRAICUkJYZhMwAsI3IMvBgBAG/tVRi8GAICWeXghUAwAxS+b
IRA8QMoWNqAvBgBAhMKqji8GABUAFb4FFcgFLBXcBBUQFQYVBhwYCEC1L0rELwYAGAhAfAS0PC8GABYAKAhAtS9KxC8GABgIQHwE
tDwvBgAREQAAAN8C9F4BAwAAANwEAQlNAAIIGECggIEDCBIoWMCggYMHECJImEChgoULGDJo2MChg4cPIEKIGEGihIkTKFKoWMGi
hYsXMGLImEGjho0bOHLo2MGjh48fQIIIGUKkiJEjSJIoWcKkiZMnUKJImUKlipUrWLJo2cKli5cvYMKIGUOmjJkzaNKoWcOmjZs3
cOLImUOnjp07ePLo2cOnj58/gAIJGkSokKFDiBIpWsSokaNHkCJJmkSpkqVLmDJp2sSpk6dPoEKJGkWqlKlTqFKpWsWqlatXsGLJ
mkWrlq1buHLp2sWrl69fwIIJG0asmLFjyJIpW8asmbNn0KJJm0atmrVr2LJp28atm7dv4MKJG0eunLlz6NKpW8eunbt38OLJm0ev
nr17+PLp28evn79/AAMKHEiwoMGDCBMqXMiwocOHECNKnEixosWLGDNq3Mixo8ePIEOKHEmypMmTKFOqXMmyJQAAFQQZTDUAGAZz
Y2hlbWEVBgAVBCUCGAhldmVudF9pZAAVBCUCGAd1c2VyX2lkABUEJQIYCGV2ZW50X3RzJRRMjBIcLAAAAAAAFtwEGRwZPCYAHBUE
GTUABhAZGAhldmVudF9pZBUCFtwEFsQsFv4ZJroTJggcGAhunQAAAAAAABgIQZwAAAAAAAAWACgIbp0AAAAAAAAYCEGcAAAAAAAA
EREAGSwVBBUAFQIAFQAVEBUCADwpBhkmANwEAAAAJgAcFQQZNQAGEBkYB3VzZXJfaWQVAhbcBBbeDBaWCSa2HiaGGhwYCD4AAAAA
AAAAGAgBAAAAAAAAABYAKAg+AAAAAAAAABgIAQAAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYA3AQAAAAmABwVBBk1AAYQ
GRgIZXZlbnRfdHMVAhbcBBbELBboKCa4RSacIxwYCEC1L0rELwYAGAhAfAS0PC8GABYAKAhAtS9KxC8GABgIQHwEtDwvBgAREQAZ
LBUEFQAVAgAVABUQFQIAPCkGGSYA3AQAAAAW5mUW3AQmCBb8SwAZHBgMQVJST1c6c2NoZW1hGMwCLy8vLy8vQUFBQUFRQUFBQUFB
QUtBQXdBQmdBRkFBZ0FDZ0FBQUFBQkJBQU1BQUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFBQU1BQUFDQUFBQUFQQUFBQUFRQUFBQ2Mv
Ly8vQUFBQkNoQUFBQUFnQUFBQUJBQUFBQUFBQUFBSUFBQUFaWFpsYm5SZmRITUFBQVlBQ0FBR0FBWUFBQUFBQUFJQTBQLy8vd0FB
QVFJUUFBQUFHQUFBQUFRQUFBQUFBQUFBQndBQUFIVnpaWEpmYVdRQXZQLy8vd0FBQUFGQUFBQUFFQUFVQUFnQUJnQUhBQXdBQUFB
UUFCQUFBQUFBQUFFQ0VBQUFBQ1FBQUFBRUFBQUFBQUFBQUFnQUFBQmxkbVZ1ZEY5cFpBQUFBQUFJQUF3QUNBQUhBQWdBQUFBQUFB
QUJRQUFBQUFBQUFBQT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRk8HAAAHAAAHAAAADQDAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "014-utc-dates-in-india-report"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

app_events = spark.read.parquet((_data_dir / "app_events.parquet").resolve().as_posix())
app_events.createOrReplaceTempView("app_events")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'event_date': 'a4d0a78a7e1497f917869b4ccd05687f18917055ce1aff7bcc8e8acd4dddd80a',
                         'events': '2f03c825413b5cdca5692ca5e9b8ce003012ff04d2d02b791dd4c18cd05a2069',
                         'users': '8f1f0d9286cffc22f941b1cd011f2baa34d27ea70e036fb23f9da4aaf929e8a4'},
 'columns': [['event_date', 'date'], ['events', 'bigint'], ['users', 'bigint']],
 'fingerprint': '9777c25a068504f37825da446b90f93ddc3e6767486716b61975b8954f4b8689',
 'order_matters': False,
 'row_count': 7}

check = make_check(EXPECTED)

## The situation

The app stores every event with a UTC timestamp. The India business team gets a daily
report: events and active users per day.

An AI assistant wrote the job in the answer cell. The weekly totals match the raw
event count exactly, so it was signed off. Then the support team in Pune noticed
that activity from just after midnight keeps showing up under the previous day.

Find the bug and fix it.

## Input tables

### `app_events`

One row per event.

| Column | Type | Description |
|---|---|---|
| `event_id` | BIGINT | Unique per event. |
| `user_id` | BIGINT | The user who triggered the event. |
| `event_ts` | TIMESTAMP_NTZ | When the event happened, in UTC. |

## Requirement

Return events and active users per day, where a day is a calendar day in India.

- India Standard Time (IST) is UTC+05:30, all year. There is no daylight saving.
- `event_date` is the date of the event in IST. An event at 18:30:00 UTC is at
  00:00:00 IST on the next day.
- `events` is the number of events on that date.
- `users` is the number of different users with at least one event on that date.

## Expected output

One row per India calendar day that has events.

| Column | Type | Description |
|---|---|---|
| `event_date` | DATE | Date in IST. |
| `events` | BIGINT | |
| `users` | BIGINT | Distinct users. |

In [ ]:
app_events.show(5)

## Your answer

The cell below is the AI-written code. It runs, but its output is wrong. Find the bug and fix it. The tables are available as DataFrames and as views: `app_events`.

In [ ]:
from pyspark.sql import functions as F


def solve(spark):
    """Daily events and active users for the India report."""
    events = spark.table("app_events")

    # Derive the calendar date of each event
    events = events.withColumn("event_date", F.to_date("event_ts"))

    # Count events and distinct users per day
    result = events.groupBy("event_date").agg(
        F.count("*").alias("events"),
        F.countDistinct("user_id").alias("users"),
    )

    return result


result = solve(spark)
result.show()

In [ ]:
check(result)